# 09 — Field presence/absence inference

## Objective

This notebook is the final large methodological stage of the paper replication:

**Dataset-2 pond WAV → frozen segmentation → original unfiltered candidate → 20 PLP-RASTA → frozen
GMMs → final LLR verification → presence/absence vector**

Everything developed in notebooks 06B–08D is now treated as frozen.

### Frozen detector

- feature: **20 PLP-RASTA**
- one diagonal GMM per deployable species
- `M = 4`
- winning hypothesis = largest mean frame log-likelihood
- verification:

\[
LLR_k(X)
=
s_k(X)
-
\operatorname{median}_{j\neq k}s_j(X)
\]

- accept the winning hypothesis only when its final threshold is passed.

### Frozen segmentation

- consecutive **30 s** blocks;
- Table-1 species frequency bands;
- FIR used only for endpoint localization;
- **10 ms non-overlapping STE**;
- moving average **MA = 12**;
- **3 above / 3 below** endpoint persistence;
- species-specific `C` learned on Dataset 1;
- `f04` only: **20 ms symmetric endpoint padding**;
- every classified waveform is taken from the **original unfiltered recording**.

`[PAPER]` Presence in one field sample is declared when at least one call of the species is accepted.

---

## Important replication limitations

1. The paper evaluates 10 species. Our recovered `f09` labelled corpus is insufficient, so the deployed
   detector contains **9 species**. `f09` is reported as `NA`, never as biological absence.
2. The paper does not publish the segmentation constant `C` or FIR order. Our final values are documented
   reconstruction/development decisions.
3. The paper ultimately used m-FCC/M=64; our recording-separated experiments selected
   **PLP-RASTA/M=4**. This is a documented project divergence, not a claim about the paper.
4. The paper manually labelled 18 field samples with complete 10-species vectors. Unless those exact
   complete labels are recoverable locally, this notebook **does not invent test metrics** from partial
   notes.
5. Thresholds and segmentation parameters are not retuned on ponds. The field corpus is finally allowed
   to be what a test corpus is supposed to be, a concept humanity occasionally remembers.

In [2]:
from pathlib import Path
import json
import os
import re
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf

from scipy.linalg import solve_toeplitz
from scipy.signal import fftconvolve, firwin, lfilter

try:
    from dotenv import load_dotenv
except ImportError as exc:
    raise ImportError(
        "Notebook 09 needs python-dotenv to resolve PONDS_CALL_DIR."
    ) from exc

pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 220)
pd.set_option("display.max_columns", 140)

CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

MODEL_DIR = (
    REPO_ROOT
    / "models"
    / "gmm_robust_final"
)

SEGMENTATION_DIR = (
    REPO_ROOT
    / "data"
    / "processed"
    / "candidate_call_segmentation"
)

FINAL_SEGMENTATION_CONFIG_PATH = (
    SEGMENTATION_DIR
    / "final_segmentation_configuration.json"
)

FINAL_DETECTOR_MANIFEST_PATH = (
    MODEL_DIR
    / "final_detector_manifest.json"
)

THRESHOLDS_PATH = (
    MODEL_DIR
    / "thresholds.json"
)

RESULTS_DIR = (
    REPO_ROOT
    / "data"
    / "processed"
    / "field_presence_absence"
)

PER_FILE_DIR = (
    RESULTS_DIR
    / "per_file"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PER_FILE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for path in (
    FINAL_SEGMENTATION_CONFIG_PATH,
    FINAL_DETECTOR_MANIFEST_PATH,
    THRESHOLDS_PATH,
):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing frozen project artifact: {path}"
        )

load_dotenv(
    REPO_ROOT / ".env"
)

pond_value = os.environ.get(
    "PONDS_CALL_DIR"
)

if not pond_value:
    raise RuntimeError(
        "PONDS_CALL_DIR is not defined in .env. "
        "Dataset 2 is the private pond-recordings collection."
    )

PONDS_CALL_DIR = Path(
    pond_value
).expanduser()

if not PONDS_CALL_DIR.is_dir():
    raise FileNotFoundError(
        f"PONDS_CALL_DIR does not exist: {PONDS_CALL_DIR}"
    )

print("Repository root  :", REPO_ROOT)
print("Dataset 2 / ponds:", PONDS_CALL_DIR)
print("Frozen models    :", MODEL_DIR)
print("Frozen segmenter :", FINAL_SEGMENTATION_CONFIG_PATH)
print("Results          :", RESULTS_DIR)

Repository root  : C:\Users\sofia\Projects\PUCE\frog-presence-replication
Dataset 2 / ponds: C:\Users\sofia\Projects\PUCE\frog-presence-replication\data\pond-recordings
Frozen models    : C:\Users\sofia\Projects\PUCE\frog-presence-replication\models\gmm_robust_final
Frozen segmenter : C:\Users\sofia\Projects\PUCE\frog-presence-replication\data\processed\candidate_call_segmentation\final_segmentation_configuration.json
Results          : C:\Users\sofia\Projects\PUCE\frog-presence-replication\data\processed\field_presence_absence


## 2. Execution controls

A complete scan can be computationally expensive because every 30-second block is examined through nine
species-specific frequency bands.

Per-file checkpointing is enabled. If execution is interrupted, rerunning the notebook skips files that
already have a successful saved result.

For the paper-level run, leave `MAX_FIELD_FILES = None`.

In [3]:
RUN_FIELD_SCAN = True

# None = all eligible Dataset-2 WAV files.
MAX_FIELD_FILES = None

# Reuse successful per-file checkpoints.
RESUME_COMPLETED_FILES = True

# Build global candidate/detection tables at the end.
# Turn off only if local RAM is unusually constrained.
BUILD_GLOBAL_EVENT_TABLES = True

# Print progress every N 30-second blocks.
BLOCK_PROGRESS_EVERY = 5

print("RUN_FIELD_SCAN         :", RUN_FIELD_SCAN)
print("MAX_FIELD_FILES        :", MAX_FIELD_FILES)
print("RESUME_COMPLETED_FILES :", RESUME_COMPLETED_FILES)

RUN_FIELD_SCAN         : True
MAX_FIELD_FILES        : None
RESUME_COMPLETED_FILES : True


## 3. Load and verify the frozen detector and segmenter

In [4]:
with open(
    FINAL_DETECTOR_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as f:
    detector_manifest = json.load(f)

with open(
    THRESHOLDS_PATH,
    "r",
    encoding="utf-8",
) as f:
    THRESHOLDS = json.load(f)

with open(
    FINAL_SEGMENTATION_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as f:
    segmentation_config = json.load(f)

CLASSES = tuple(
    detector_manifest[
        "classes"
    ]
)

if detector_manifest[
    "detector_status"
] != "frozen_after_07C_threshold_promotion":
    raise RuntimeError(
        "Final detector manifest is not in the expected frozen state."
    )

if detector_manifest[
    "feature"
] != "20_PLP_RASTA":
    raise RuntimeError(
        "Notebook 09 expects final feature 20_PLP_RASTA."
    )

if int(
    detector_manifest[
        "n_components"
    ]
) != 4:
    raise RuntimeError(
        "Notebook 09 expects final GMM order M=4."
    )

if set(
    THRESHOLDS
) != set(
    CLASSES
):
    raise RuntimeError(
        "Threshold classes do not match the final detector manifest."
    )

if segmentation_config.get(
    "frozen_for_field_inference"
) is not True:
    raise RuntimeError(
        "Segmentation configuration is not frozen for field inference."
    )

if segmentation_config.get(
    "do_not_retune_on_ponds"
) is not True:
    raise RuntimeError(
        "Frozen segmentation provenance is missing the no-retuning flag."
    )

C_BY_CODE = {
    code: float(value)
    for code, value
    in segmentation_config[
        "selection"
    ][
        "C_by_code"
    ].items()
}

PADDING_MS_BY_CODE = {
    code: int(value)
    for code, value
    in segmentation_config[
        "endpoint_padding_ms_per_side"
    ].items()
}

BANDS_HZ = {
    code: tuple(
        float(v)
        for v
        in segmentation_config[
            "paper_frontend"
        ][
            "species_frequency_bands_hz"
        ][
            code
        ]
    )
    for code in CLASSES
}

if set(C_BY_CODE) != set(CLASSES):
    raise RuntimeError("C configuration does not match detector classes.")

if set(PADDING_MS_BY_CODE) != set(CLASSES):
    raise RuntimeError("Padding configuration does not match detector classes.")

print("Classes :", CLASSES)
print("Feature :", detector_manifest["feature"])
print("GMM M   :", detector_manifest["n_components"])

display(
    pd.DataFrame([
        {
            "code": code,
            "threshold": float(
                THRESHOLDS[
                    code
                ]
            ),
            "C": C_BY_CODE[
                code
            ],
            "padding_ms_per_side": PADDING_MS_BY_CODE[
                code
            ],
            "low_hz": BANDS_HZ[
                code
            ][0],
            "high_hz": BANDS_HZ[
                code
            ][1],
        }
        for code in CLASSES
    ]).round(4)
)

Classes : ('f01', 'f02', 'f03', 'f04', 'f05', 'f06', 'f07', 'f08', 'f10')
Feature : 20_PLP_RASTA
GMM M   : 4


,code,threshold,C,padding_ms_per_side,low_hz,high_hz
0,f01,8.0751,4.00,0,1660.0,3100.0
1,f02,3.7012,2.00,0,2300.0,3390.0
2,f03,6.9382,4.00,0,1300.0,1530.0
3,f04,5.0647,10.00,20,1630.0,3900.0
4,f05,5.5535,2.50,0,1680.0,3260.0
5,f06,5.2310,4.00,0,1000.0,2500.0
6,f07,5.7501,4.00,0,500.0,2720.0
7,f08,7.1894,1.75,0,800.0,1700.0
8,f10,9.0146,4.00,0,430.0,3140.0


## 4. Load the frozen GMMs

No fitting occurs anywhere in this notebook.

In [5]:
MODELS = {}

for code_name in CLASSES:
    model_path = (
        MODEL_DIR
        / f"{code_name}.joblib"
    )

    if not model_path.exists():
        raise FileNotFoundError(
            f"Missing frozen GMM: {model_path}"
        )

    model = joblib.load(
        model_path
    )

    if model.covariance_type != "diag":
        raise RuntimeError(
            f"{code_name} is not a diagonal-covariance GMM."
        )

    if int(
        model.n_components
    ) != 4:
        raise RuntimeError(
            f"{code_name} does not use M=4."
        )

    MODELS[
        code_name
    ] = model

print("Loaded frozen models:", sorted(MODELS))

Loaded frozen models: ['f01', 'f02', 'f03', 'f04', 'f05', 'f06', 'f07', 'f08', 'f10']


## 5. Inventory Dataset 2

`[PAPER]` The test material is described as 141 nominally 10-minute samples digitized at 48 kHz.

The recovered collection is audited rather than forced into that description. A WAV is eligible when:

- it can be read by libsndfile;
- sample rate is 48 kHz;
- subtype is PCM.

Duration is reported as observed. A non-600-second file is not silently discarded.

In [6]:
def infer_pond(path):
    text = " ".join(
        part.lower()
        for part in path.parts
    )

    p1_patterns = (
        r"\bp1\b",
        r"pond\s*1",
        r"poza\s*1",
    )

    p2_patterns = (
        r"\bp2\b",
        r"pond\s*2",
        r"poza\s*2",
    )

    if any(
        re.search(
            pattern,
            text,
        )
        for pattern
        in p1_patterns
    ):
        return "P1"

    if any(
        re.search(
            pattern,
            text,
        )
        for pattern
        in p2_patterns
    ):
        return "P2"

    return "unknown"


def infer_month(path):
    text = " ".join(
        part.lower()
        for part in path.parts
    )

    month_patterns = {
        "February": (
            r"feb",
            r"febrero",
            r"02[_\- ]",
        ),
        "April": (
            r"apr",
            r"abril",
            r"04[_\- ]",
        ),
        "July": (
            r"jul",
            r"julio",
            r"07[_\- ]",
        ),
        "August": (
            r"aug",
            r"agosto",
            r"08[_\- ]",
        ),
        "September": (
            r"sep",
            r"sept",
            r"septiembre",
            r"09[_\- ]",
        ),
    }

    for month, patterns in month_patterns.items():
        if any(
            re.search(
                pattern,
                text,
            )
            for pattern
            in patterns
        ):
            return month

    return "unknown"


def sample_id_from_path(path):
    relative = path.relative_to(
        PONDS_CALL_DIR
    ).with_suffix("")

    return "__".join(
        relative.parts
    )


inventory_rows = []

wav_paths = sorted(
    path
    for path
    in PONDS_CALL_DIR.rglob("*")
    if path.suffix.lower() == ".wav"
)

for path in wav_paths:
    row = {
        "sample_id":
            sample_id_from_path(
                path
            ),
        "path":
            str(path),
        "relative_path":
            str(
                path.relative_to(
                    PONDS_CALL_DIR
                )
            ),
        "pond":
            infer_pond(
                path
            ),
        "month":
            infer_month(
                path
            ),
        "readable":
            False,
        "sample_rate":
            np.nan,
        "frames":
            np.nan,
        "duration_seconds":
            np.nan,
        "format":
            None,
        "subtype":
            None,
        "eligible":
            False,
        "reason":
            None,
    }

    try:
        info = sf.info(
            path
        )

        row.update({
            "readable":
                True,
            "sample_rate":
                int(
                    info.samplerate
                ),
            "frames":
                int(
                    info.frames
                ),
            "duration_seconds":
                float(
                    info.duration
                ),
            "format":
                str(
                    info.format
                ),
            "subtype":
                str(
                    info.subtype
                ),
        })

        reasons = []

        if int(
            info.samplerate
        ) != 48_000:
            reasons.append(
                f"sample_rate={info.samplerate}"
            )

        if not str(
            info.subtype
        ).startswith(
            "PCM_"
        ):
            reasons.append(
                f"subtype={info.subtype}"
            )

        row[
            "eligible"
        ] = (
            len(reasons)
            == 0
        )

        row[
            "reason"
        ] = (
            "; ".join(
                reasons
            )
            if reasons
            else "ok"
        )

    except Exception as exc:
        row[
            "reason"
        ] = repr(
            exc
        )

    inventory_rows.append(
        row
    )

field_inventory = pd.DataFrame(
    inventory_rows
)

if (
    not field_inventory.empty
    and not field_inventory[
        "sample_id"
    ].is_unique
):
    raise RuntimeError(
        "Dataset-2 sample identifiers are not unique."
    )

field_inventory.to_csv(
    RESULTS_DIR
    / "field_inventory.csv",
    index=False,
)

print("WAV files found :", len(field_inventory))
print(
    "Eligible WAVs   :",
    int(
        field_inventory[
            "eligible"
        ].sum()
    )
    if len(field_inventory)
    else 0,
)

if len(
    field_inventory
):
    display(
        field_inventory[
            [
                "sample_id",
                "pond",
                "month",
                "sample_rate",
                "duration_seconds",
                "subtype",
                "eligible",
                "reason",
            ]
        ]
        .head(
            30
        )
        .round(3)
    )

eligible_inventory = (
    field_inventory[
        field_inventory[
            "eligible"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

if len(
    eligible_inventory
) != 141:
    warnings.warn(
        f"Paper reports 141 nominal field samples; "
        f"recovered eligible inventory contains {len(eligible_inventory)}."
    )

if len(
    eligible_inventory
):
    print(
        "Eligible duration:",
        round(
            eligible_inventory[
                "duration_seconds"
            ].sum()
            / 3600.0,
            3,
        ),
        "hours",
    )

WAV files found : 1680
Eligible WAVs   : 546


,sample_id,pond,month,sample_rate,duration_seconds,subtype,eligible,reason
0,qcaz-cc106__A__._cc106A-Poza1_SP1,P1,unknown,NaN,NaN,NaN,False,"LibsndfileError(1, ""Error opening 'C:\\\\Users..."
1,qcaz-cc106__A__._cc106A-Poza1_SP3,P1,unknown,NaN,NaN,NaN,False,"LibsndfileError(1, ""Error opening 'C:\\\\Users..."
2,qcaz-cc106__A__._cc106A-Poza2_SP2,P2,unknown,NaN,NaN,NaN,False,"LibsndfileError(1, ""Error opening 'C:\\\\Users..."
3,qcaz-cc106__A__._cc106A-Poza2_SP2A,P2,unknown,NaN,NaN,NaN,False,"LibsndfileError(1, ""Error opening 'C:\\\\Users..."
4,qcaz-cc106__A__._cc106A-Poza2_SP2B,P2,unknown,NaN,NaN,NaN,False,"LibsndfileError(1, ""Error opening 'C:\\\\Users..."
5,qcaz-cc106__A__._cc106A,unknown,unknown,NaN,NaN,NaN,False,"LibsndfileError(1, ""Error opening 'C:\\\\Users..."
6,qcaz-cc106__A__._cc106A_SP1,unknown,unknown,NaN,NaN,NaN,False,"LibsndfileError(1, ""Error opening 'C:\\\\Users..."
7,qcaz-cc106__A__cc106A-Poza1_SP1,P1,unknown,44100.0,633.905,GSM610,False,sample_rate=44100; subtype=GSM610
8,qcaz-cc106__A__cc106A-Poza1_SP3,P1,unknown,44100.0,598.886,GSM610,False,sample_rate=44100; subtype=GSM610
9,qcaz-cc106__A__cc106A-Poza2_SP2,P2,unknown,44100.0,496.718,GSM610,False,sample_rate=44100; subtype=GSM610


Eligible duration: 51.373 hours


C:\Users\sofia\AppData\Local\Temp\ipykernel_14216\1887138050.py:293: UserWarning: Paper reports 141 nominal field samples; recovered eligible inventory contains 546.
  warnings.warn(


## 6. Frozen segmentation functions

In [7]:
BLOCK_SECONDS = float(
    segmentation_config[
        "paper_frontend"
    ][
        "block_seconds"
    ]
)

STE_FRAME_SECONDS = float(
    segmentation_config[
        "paper_frontend"
    ][
        "ste_frame_seconds"
    ]
)

STE_MA_LENGTH = int(
    segmentation_config[
        "paper_frontend"
    ][
        "moving_average_length"
    ]
)

ENDPOINT_RUN = int(
    segmentation_config[
        "paper_frontend"
    ][
        "endpoint_run_length"
    ]
)

FIR_NUMTAPS = int(
    segmentation_config[
        "replication_decisions"
    ][
        "fir_numtaps"
    ]
)


def to_mono(audio):
    audio = np.asarray(
        audio,
        dtype=np.float64,
    )

    if audio.ndim == 1:
        return audio

    if audio.ndim == 2:
        return audio.mean(
            axis=1
        )

    raise ValueError(
        f"Unsupported audio shape: {audio.shape}"
    )


_FIR_CACHE = {}


def bandpass_for_segmentation(
    audio,
    sr,
    low_hz,
    high_hz,
):
    key = (
        int(sr),
        float(low_hz),
        float(high_hz),
        FIR_NUMTAPS,
    )

    if key not in _FIR_CACHE:
        _FIR_CACHE[
            key
        ] = firwin(
            numtaps=FIR_NUMTAPS,
            cutoff=[
                low_hz,
                high_hz,
            ],
            pass_zero=False,
            fs=sr,
        )

    return fftconvolve(
        audio,
        _FIR_CACHE[
            key
        ],
        mode="same",
    )


def short_time_energy(
    audio,
    sr,
):
    frame_samples = int(
        round(
            sr
            * STE_FRAME_SECONDS
        )
    )

    n_frames = (
        len(audio)
        // frame_samples
    )

    if n_frames == 0:
        return (
            np.empty(
                0,
                dtype=float,
            ),
            frame_samples,
        )

    trimmed = audio[
        :
        n_frames
        * frame_samples
    ]

    frames = trimmed.reshape(
        n_frames,
        frame_samples,
    )

    energy = np.sum(
        frames
        * frames,
        axis=1,
    )

    return (
        energy,
        frame_samples,
    )


def moving_average_same(
    values,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    if len(values) == 0:
        return values.copy()

    kernel = (
        np.ones(
            STE_MA_LENGTH,
            dtype=float,
        )
        / STE_MA_LENGTH
    )

    return np.convolve(
        values,
        kernel,
        mode="same",
    )


def energy_to_db(
    energy,
):
    return (
        10.0
        * np.log10(
            np.maximum(
                np.asarray(
                    energy,
                    dtype=float,
                ),
                np.finfo(
                    float
                ).tiny,
            )
        )
    )


def endpoint_threshold_db(
    ste_db,
    C,
):
    ste_db = np.asarray(
        ste_db,
        dtype=float,
    )

    if len(ste_db) == 0:
        return np.nan

    mean_db = float(
        ste_db.mean()
    )

    max_db = float(
        ste_db.max()
    )

    return (
        mean_db
        + (
            max_db
            - mean_db
        )
        / C
    )


def find_endpoint_runs(
    ste_db,
    threshold_db,
):
    ste_db = np.asarray(
        ste_db,
        dtype=float,
    )

    segments = []

    in_segment = False
    start_idx = None

    i = 0

    while (
        i
        <= len(ste_db)
        - ENDPOINT_RUN
    ):
        current = ste_db[
            i:
            i
            + ENDPOINT_RUN
        ]

        if not in_segment:
            if np.all(
                current
                > threshold_db
            ):
                start_idx = i
                in_segment = True
                i += ENDPOINT_RUN
                continue

        else:
            if np.all(
                current
                < threshold_db
            ):
                end_idx = i

                if (
                    start_idx
                    is not None
                    and end_idx
                    > start_idx
                ):
                    segments.append(
                        (
                            start_idx,
                            end_idx,
                        )
                    )

                start_idx = None
                in_segment = False
                i += ENDPOINT_RUN
                continue

        i += 1

    if (
        in_segment
        and start_idx
        is not None
    ):
        segments.append(
            (
                start_idx,
                len(
                    ste_db
                ),
            )
        )

    return segments


def merge_intervals(
    intervals,
):
    intervals = sorted(
        (
            int(start),
            int(end),
        )
        for start, end
        in intervals
        if end > start
    )

    if not intervals:
        return []

    merged = [
        list(
            intervals[0]
        )
    ]

    for start, end in intervals[
        1:
    ]:
        previous = merged[
            -1
        ]

        if start <= previous[1]:
            previous[1] = max(
                previous[1],
                end,
            )
        else:
            merged.append(
                [
                    start,
                    end,
                ]
            )

    return [
        (
            start,
            end,
        )
        for start, end
        in merged
    ]

## 7. Exact 20-dimensional PLP-RASTA frontend used by the frozen models

In [8]:
PRE_EMPHASIS = 0.99
FEATURE_FRAME_SECONDS = 0.020
FEATURE_OVERLAP = 0.75
NFFT = 1024

N_BARK_FILTERS = 24
PLP_ORDER = 20
N_PLP_CEPSTRA = 20


def pre_emphasise(audio):
    audio = np.asarray(
        audio,
        dtype=np.float64,
    )

    if len(audio) == 0:
        return audio.copy()

    out = np.empty_like(
        audio
    )

    out[0] = audio[0]

    out[1:] = (
        audio[1:]
        - PRE_EMPHASIS
        * audio[:-1]
    )

    return out


def frame_audio(
    audio,
    sr,
):
    frame_length = int(
        round(
            FEATURE_FRAME_SECONDS
            * sr
        )
    )

    hop_length = int(
        round(
            FEATURE_FRAME_SECONDS
            * (
                1.0
                - FEATURE_OVERLAP
            )
            * sr
        )
    )

    if len(audio) < frame_length:
        return np.empty(
            (
                0,
                frame_length,
            ),
            dtype=float,
        )

    n_frames = (
        1
        + (
            len(audio)
            - frame_length
        )
        // hop_length
    )

    starts = (
        np.arange(
            n_frames
        )
        * hop_length
    )

    return np.stack([
        audio[
            start:
            start
            + frame_length
        ]
        for start
        in starts
    ])


def frame_power_spectrum(
    audio,
    sr,
):
    emphasized = pre_emphasise(
        audio
    )

    frames = frame_audio(
        emphasized,
        sr,
    )

    if len(frames) == 0:
        return (
            np.empty(
                (
                    0,
                    NFFT // 2 + 1,
                )
            ),
            frames,
        )

    window = np.hamming(
        frames.shape[1]
    )

    spectrum = np.fft.rfft(
        frames
        * window[
            None,
            :
        ],
        n=NFFT,
        axis=1,
    )

    power = (
        np.abs(
            spectrum
        ) ** 2
    ) / NFFT

    return (
        power,
        frames,
    )


def hz_to_bark(hz):
    hz = np.asarray(
        hz,
        dtype=float,
    )

    return (
        13.0
        * np.arctan(
            0.00076
            * hz
        )
        + 3.5
        * np.arctan(
            (
                hz
                / 7500.0
            ) ** 2
        )
    )


def bark_to_hz(bark):
    bark = np.asarray(
        bark,
        dtype=float,
    )

    hz_grid = np.linspace(
        0.0,
        24_000.0,
        200_001,
    )

    bark_grid = hz_to_bark(
        hz_grid
    )

    return np.interp(
        bark,
        bark_grid,
        hz_grid,
    )


_BARK_BANK_CACHE = {}


def bark_filterbank(
    sr,
    n_fft=NFFT,
    n_filters=N_BARK_FILTERS,
):
    key = (
        int(sr),
        int(n_fft),
        int(n_filters),
    )

    if key in _BARK_BANK_CACHE:
        return _BARK_BANK_CACHE[
            key
        ]

    max_bark = float(
        hz_to_bark(
            sr / 2.0
        )
    )

    bark_points = np.linspace(
        0.0,
        max_bark,
        n_filters
        + 2,
    )

    hz_points = bark_to_hz(
        bark_points
    )

    fft_freqs = np.fft.rfftfreq(
        n_fft,
        d=1.0 / sr,
    )

    bank = np.zeros(
        (
            n_filters,
            len(
                fft_freqs
            ),
        ),
        dtype=float,
    )

    centers = []

    for i in range(
        n_filters
    ):
        left = hz_points[
            i
        ]

        center = hz_points[
            i + 1
        ]

        right = hz_points[
            i + 2
        ]

        centers.append(
            center
        )

        left_mask = (
            (
                fft_freqs
                >= left
            )
            & (
                fft_freqs
                <= center
            )
        )

        right_mask = (
            (
                fft_freqs
                >= center
            )
            & (
                fft_freqs
                <= right
            )
        )

        if center > left:
            bank[
                i,
                left_mask,
            ] = (
                fft_freqs[
                    left_mask
                ]
                - left
            ) / (
                center
                - left
            )

        if right > center:
            bank[
                i,
                right_mask,
            ] = (
                right
                - fft_freqs[
                    right_mask
                ]
            ) / (
                right
                - center
            )

    result = (
        bank,
        np.asarray(
            centers
        ),
    )

    _BARK_BANK_CACHE[
        key
    ] = result

    return result


def rasta_filter_log_spectrum(
    log_spectrum,
):
    numerator = np.array([
        0.2,
        0.1,
        0.0,
        -0.1,
        -0.2,
    ])

    denominator = np.array([
        1.0,
        -0.94,
    ])

    return lfilter(
        numerator,
        denominator,
        log_spectrum,
        axis=0,
    )


def equal_loudness_curve(
    freq_hz,
):
    f2 = np.asarray(
        freq_hz,
        dtype=float,
    ) ** 2

    numerator = (
        (
            f2
            + 1.6e5
        )
        * (
            f2 ** 2
        )
    )

    denominator = (
        (
            f2
            + 1.44e6
        )
        * (
            (
                f2
                + 9.61e6
            ) ** 2
        )
    )

    curve = (
        numerator
        / np.maximum(
            denominator,
            np.finfo(
                float
            ).eps,
        )
    )

    return np.maximum(
        curve,
        np.finfo(
            float
        ).eps,
    )


def auditory_spectrum_to_lpc(
    spectrum,
    order=PLP_ORDER,
):
    spectrum = np.asarray(
        spectrum,
        dtype=float,
    )

    n_ifft = 2 * (
        len(
            spectrum
        )
        - 1
    )

    autocorr = np.fft.irfft(
        spectrum,
        n=n_ifft,
    )

    autocorr = np.real(
        autocorr[
            :
            order
            + 1
        ]
    )

    autocorr[0] = max(
        autocorr[0],
        1e-10,
    )

    r0 = autocorr[0]

    autocorr[0] += (
        1e-6
        * abs(
            r0
        )
    )

    try:
        coeff = solve_toeplitz(
            (
                autocorr[
                    :
                    order
                ],
                autocorr[
                    :
                    order
                ],
            ),
            -autocorr[
                1:
                order
                + 1
            ],
        )

    except Exception:
        coeff = np.zeros(
            order,
            dtype=float,
        )

    lpc = np.concatenate([
        [
            1.0
        ],
        coeff,
    ])

    error = float(
        autocorr[0]
        + np.dot(
            autocorr[
                1:
                order
                + 1
            ],
            coeff,
        )
    )

    return (
        lpc,
        max(
            error,
            1e-12,
        ),
    )


def lpc_to_cepstra(
    lpc,
    error,
    n_cepstra=N_PLP_CEPSTRA,
):
    order = len(
        lpc
    ) - 1

    c = np.zeros(
        n_cepstra,
        dtype=float,
    )

    c[0] = np.log(
        max(
            error,
            1e-12,
        )
    )

    for n in range(
        1,
        n_cepstra,
    ):
        total = 0.0

        k_start = max(
            1,
            n
            - order,
        )

        for k in range(
            k_start,
            n,
        ):
            a_index = (
                n
                - k
            )

            if (
                a_index
                <= order
                and k
                < len(c)
            ):
                total += (
                    (
                        k
                        / n
                    )
                    * c[k]
                    * lpc[
                        a_index
                    ]
                )

        if n <= order:
            c[n] = (
                -lpc[n]
                - total
            )
        else:
            c[n] = (
                -total
            )

    return c


def plp_rasta_20(
    audio,
    sr,
):
    power, frames = frame_power_spectrum(
        audio,
        sr,
    )

    if len(frames) == 0:
        return np.empty(
            (
                0,
                N_PLP_CEPSTRA,
            )
        )

    bark_bank, centers_hz = bark_filterbank(
        sr
    )

    critical = (
        power
        @ bark_bank.T
    )

    critical = np.maximum(
        critical,
        np.finfo(
            float
        ).eps,
    )

    log_critical = np.log(
        critical
    )

    rasta_log = (
        rasta_filter_log_spectrum(
            log_critical
        )
    )

    rasta_energy = np.exp(
        rasta_log
    )

    eql = equal_loudness_curve(
        centers_hz
    )

    post_auditory = (
        rasta_energy
        * eql[
            None,
            :
        ]
    )

    compressed = np.cbrt(
        np.maximum(
            post_auditory,
            np.finfo(
                float
            ).eps,
        )
    )

    features_out = []

    for auditory_frame in compressed:
        lpc, error = (
            auditory_spectrum_to_lpc(
                auditory_frame,
                order=PLP_ORDER,
            )
        )

        features_out.append(
            lpc_to_cepstra(
                lpc,
                error,
                n_cepstra=N_PLP_CEPSTRA,
            )
        )

    return np.asarray(
        features_out,
        dtype=float,
    )

## 8. Frozen detector scoring

In [9]:
def score_candidate_features(
    X,
):
    X = np.asarray(
        X,
        dtype=float,
    )

    if (
        X.ndim != 2
        or X.shape[1] != 20
        or len(X) == 0
        or not np.isfinite(
            X
        ).all()
    ):
        return None

    scores = {
        code_name:
            float(
                model.score(
                    X
                )
            )
        for code_name, model
        in MODELS.items()
    }

    hypothesis = max(
        scores,
        key=scores.get,
    )

    alternative_scores = [
        score
        for code_name, score
        in scores.items()
        if code_name
        != hypothesis
    ]

    llr = (
        scores[
            hypothesis
        ]
        - float(
            np.median(
                alternative_scores
            )
        )
    )

    threshold = float(
        THRESHOLDS[
            hypothesis
        ]
    )

    accepted = (
        llr
        >= threshold
    )

    return {
        "scores":
            scores,
        "hypothesis":
            hypothesis,
        "hypothesis_score":
            scores[
                hypothesis
            ],
        "llr":
            llr,
        "threshold":
            threshold,
        "accepted":
            bool(
                accepted
            ),
        "prediction":
            (
                hypothesis
                if accepted
                else None
            ),
    }

## 9. Segment one field file

The file is read in 30-second blocks for segmentation. Candidate timestamps are stored globally.

After all blocks have been segmented:

1. endpoint padding is applied;
2. intervals are merged **within their segmentation band**;
3. candidate audio is reread directly from the original WAV;
4. PLP-RASTA and the frozen detector are applied.

This avoids holding an entire 10-minute recording in RAM while still allowing the 20 ms `f04` padding
to cross a 30-second boundary.

In [10]:
def segment_field_file(
    path,
):
    path = Path(
        path
    )

    info = sf.info(
        path
    )

    sr = int(
        info.samplerate
    )

    total_samples = int(
        info.frames
    )

    block_samples = int(
        round(
            BLOCK_SECONDS
            * sr
        )
    )

    raw_candidate_rows = []

    with sf.SoundFile(
        path,
        "r",
    ) as audio_file:

        n_blocks = int(
            np.ceil(
                total_samples
                / block_samples
            )
        )

        for block_index in range(
            n_blocks
        ):
            block_start = (
                block_index
                * block_samples
            )

            audio_file.seek(
                block_start
            )

            original_block = audio_file.read(
                frames=block_samples,
                dtype="float64",
                always_2d=False,
            )

            original_block = to_mono(
                original_block
            )

            if len(
                original_block
            ) == 0:
                continue

            if (
                BLOCK_PROGRESS_EVERY
                and (
                    block_index
                    % BLOCK_PROGRESS_EVERY
                    == 0
                )
            ):
                print(
                    f"    block {block_index + 1}/{n_blocks}"
                )

            for segmentation_code in CLASSES:
                low_hz, high_hz = (
                    BANDS_HZ[
                        segmentation_code
                    ]
                )

                filtered = (
                    bandpass_for_segmentation(
                        original_block,
                        sr,
                        low_hz,
                        high_hz,
                    )
                )

                ste, frame_samples = (
                    short_time_energy(
                        filtered,
                        sr,
                    )
                )

                ste_db = energy_to_db(
                    moving_average_same(
                        ste
                    )
                )

                if len(
                    ste_db
                ) == 0:
                    continue

                C = C_BY_CODE[
                    segmentation_code
                ]

                threshold_db = (
                    endpoint_threshold_db(
                        ste_db,
                        C,
                    )
                )

                frame_segments = (
                    find_endpoint_runs(
                        ste_db,
                        threshold_db,
                    )
                )

                for (
                    start_frame,
                    end_frame,
                ) in frame_segments:
                    start_sample = (
                        block_start
                        + start_frame
                        * frame_samples
                    )

                    end_sample = min(
                        total_samples,
                        block_start
                        + end_frame
                        * frame_samples,
                    )

                    if end_sample <= start_sample:
                        continue

                    raw_candidate_rows.append({
                        "segmentation_code":
                            segmentation_code,
                        "block_index":
                            block_index,
                        "C":
                            float(
                                C
                            ),
                        "threshold_db":
                            float(
                                threshold_db
                            ),
                        "raw_start_sample":
                            int(
                                start_sample
                            ),
                        "raw_end_sample":
                            int(
                                end_sample
                            ),
                    })

    raw_candidates = pd.DataFrame(
        raw_candidate_rows
    )

    if raw_candidates.empty:
        return (
            raw_candidates,
            pd.DataFrame(),
        )

    padded_rows = []

    for segmentation_code, group in raw_candidates.groupby(
        "segmentation_code",
        sort=True,
    ):
        pad_samples = int(
            round(
                sr
                * PADDING_MS_BY_CODE[
                    segmentation_code
                ]
                / 1000.0
            )
        )

        padded_intervals = [
            (
                max(
                    0,
                    int(
                        row.raw_start_sample
                    )
                    - pad_samples,
                ),
                min(
                    total_samples,
                    int(
                        row.raw_end_sample
                    )
                    + pad_samples,
                ),
            )
            for row in group.itertuples(
                index=False
            )
        ]

        merged = merge_intervals(
            padded_intervals
        )

        for merged_index, (
            start_sample,
            end_sample,
        ) in enumerate(
            merged
        ):
            padded_rows.append({
                "segmentation_code":
                    segmentation_code,
                "candidate_index":
                    int(
                        merged_index
                    ),
                "padding_ms_per_side":
                    int(
                        PADDING_MS_BY_CODE[
                            segmentation_code
                        ]
                    ),
                "start_sample":
                    int(
                        start_sample
                    ),
                "end_sample":
                    int(
                        end_sample
                    ),
                "start_seconds":
                    start_sample
                    / sr,
                "end_seconds":
                    end_sample
                    / sr,
                "duration_seconds":
                    (
                        end_sample
                        - start_sample
                    )
                    / sr,
            })

    candidates = pd.DataFrame(
        padded_rows
    )

    return (
        raw_candidates,
        candidates,
    )

## 10. Score all candidates from one field file

In [11]:
def score_field_candidates(
    path,
    candidates,
):
    path = Path(
        path
    )

    info = sf.info(
        path
    )

    sr = int(
        info.samplerate
    )

    rows = []

    with sf.SoundFile(
        path,
        "r",
    ) as audio_file:

        for row_index, row in enumerate(
            candidates.itertuples(
                index=False
            ),
            start=1,
        ):
            start_sample = int(
                row.start_sample
            )

            end_sample = int(
                row.end_sample
            )

            audio_file.seek(
                start_sample
            )

            waveform = audio_file.read(
                frames=(
                    end_sample
                    - start_sample
                ),
                dtype="float64",
                always_2d=False,
            )

            waveform = to_mono(
                waveform
            )

            X = plp_rasta_20(
                waveform,
                sr,
            )

            decision = score_candidate_features(
                X
            )

            out = {
                "segmentation_code":
                    row.segmentation_code,
                "candidate_index":
                    int(
                        row.candidate_index
                    ),
                "padding_ms_per_side":
                    int(
                        row.padding_ms_per_side
                    ),
                "start_sample":
                    start_sample,
                "end_sample":
                    end_sample,
                "start_seconds":
                    float(
                        row.start_seconds
                    ),
                "end_seconds":
                    float(
                        row.end_seconds
                    ),
                "duration_seconds":
                    float(
                        row.duration_seconds
                    ),
                "n_plp_frames":
                    int(
                        len(X)
                    ),
                "feature_valid":
                    decision
                    is not None,
            }

            if decision is None:
                out.update({
                    "hypothesis":
                        None,
                    "hypothesis_score":
                        np.nan,
                    "llr":
                        np.nan,
                    "threshold":
                        np.nan,
                    "accepted":
                        False,
                    "prediction":
                        None,
                })

                for code_name in CLASSES:
                    out[
                        f"score_{code_name}"
                    ] = np.nan

            else:
                out.update({
                    "hypothesis":
                        decision[
                            "hypothesis"
                        ],
                    "hypothesis_score":
                        decision[
                            "hypothesis_score"
                        ],
                    "llr":
                        decision[
                            "llr"
                        ],
                    "threshold":
                        decision[
                            "threshold"
                        ],
                    "accepted":
                        decision[
                            "accepted"
                        ],
                    "prediction":
                        decision[
                            "prediction"
                        ],
                })

                for code_name in CLASSES:
                    out[
                        f"score_{code_name}"
                    ] = (
                        decision[
                            "scores"
                        ][
                            code_name
                        ]
                    )

            rows.append(
                out
            )

    return pd.DataFrame(
        rows
    )

## 11. Presence aggregation

For every modelled species:

- `1` = at least one accepted detection in the sample;
- `0` = no accepted detection.

For `f09`:

- `NA` = no deployable model.

That distinction matters. `NA` is epistemic limitation, not ecological absence.

In [12]:
PAPER_CLASS_ORDER = (
    "f01",
    "f02",
    "f03",
    "f04",
    "f05",
    "f06",
    "f07",
    "f08",
    "f09",
    "f10",
)


def presence_from_detections(
    sample_id,
    detections,
):
    row = {
        "sample_id":
            sample_id,
    }

    accepted_predictions = set()

    if (
        not detections.empty
        and "accepted"
        in detections.columns
    ):
        accepted_predictions = set(
            detections.loc[
                detections[
                    "accepted"
                ].astype(bool),
                "prediction",
            ]
            .dropna()
            .astype(str)
        )

    for code_name in PAPER_CLASS_ORDER:
        column = (
            f"presence_{code_name}"
        )

        if code_name not in CLASSES:
            row[
                column
            ] = pd.NA
        else:
            row[
                column
            ] = int(
                code_name
                in accepted_predictions
            )

    return row

## 12. Deduplicate accepted detections for audit summaries

The same acoustic event can be proposed by several overlapping segmentation bands. Raw decisions are
preserved, but a second table merges overlapping **accepted detections of the same predicted species**.

Presence/absence itself is unaffected either way because one accepted call is sufficient.

In [13]:
def deduplicate_accepted_detections(
    detections,
):
    if detections.empty:
        return pd.DataFrame()

    accepted = (
        detections[
            detections[
                "accepted"
            ].astype(bool)
            & detections[
                "prediction"
            ].notna()
        ]
        .copy()
    )

    if accepted.empty:
        return pd.DataFrame()

    rows = []

    for prediction, group in accepted.groupby(
        "prediction",
        sort=True,
    ):
        group = group.sort_values(
            [
                "start_sample",
                "end_sample",
            ]
        )

        current = None

        for row in group.itertuples(
            index=False
        ):
            if current is None:
                current = {
                    "prediction":
                        prediction,
                    "start_sample":
                        int(
                            row.start_sample
                        ),
                    "end_sample":
                        int(
                            row.end_sample
                        ),
                    "best_llr":
                        float(
                            row.llr
                        ),
                    "threshold":
                        float(
                            row.threshold
                        ),
                    "segmentation_codes":
                        {
                            str(
                                row.segmentation_code
                            )
                        },
                    "raw_detection_count":
                        1,
                }

                continue

            if int(
                row.start_sample
            ) <= current[
                "end_sample"
            ]:
                current[
                    "end_sample"
                ] = max(
                    current[
                        "end_sample"
                    ],
                    int(
                        row.end_sample
                    ),
                )

                current[
                    "best_llr"
                ] = max(
                    current[
                        "best_llr"
                    ],
                    float(
                        row.llr
                    ),
                )

                current[
                    "segmentation_codes"
                ].add(
                    str(
                        row.segmentation_code
                    )
                )

                current[
                    "raw_detection_count"
                ] += 1

            else:
                rows.append(
                    current
                )

                current = {
                    "prediction":
                        prediction,
                    "start_sample":
                        int(
                            row.start_sample
                        ),
                    "end_sample":
                        int(
                            row.end_sample
                        ),
                    "best_llr":
                        float(
                            row.llr
                        ),
                    "threshold":
                        float(
                            row.threshold
                        ),
                    "segmentation_codes":
                        {
                            str(
                                row.segmentation_code
                            )
                        },
                    "raw_detection_count":
                        1,
                }

        if current is not None:
            rows.append(
                current
            )

    out = pd.DataFrame(
        rows
    )

    if out.empty:
        return out

    out[
        "segmentation_codes"
    ] = out[
        "segmentation_codes"
    ].apply(
        lambda values:
            ",".join(
                sorted(
                    values
                )
            )
    )

    return out

## 13. Per-file checkpoint paths

In [17]:
def safe_file_stem(
    sample_id,
):
    return re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        sample_id,
    )


def checkpoint_paths(
    sample_id,
):
    stem = safe_file_stem(
        sample_id
    )

    return {
        "raw_candidates":
            PER_FILE_DIR
            / f"{stem}__raw_candidates.csv",
        "candidates":
            PER_FILE_DIR
            / f"{stem}__candidates.csv",
        "detections":
            PER_FILE_DIR
            / f"{stem}__detections.csv",
        "accepted_deduplicated":
            PER_FILE_DIR
            / f"{stem}__accepted_deduplicated.csv",
        "presence":
            PER_FILE_DIR
            / f"{stem}__presence.csv",
        "run":
            PER_FILE_DIR
            / f"{stem}__run.json",
    }


def checkpoint_complete(
    sample_id,
):
    paths = checkpoint_paths(
        sample_id
    )

    if not paths[
        "run"
    ].exists():
        return False

    try:
        with open(
            paths[
                "run"
            ],
            "r",
            encoding="utf-8",
        ) as f:
            run = json.load(
                f
            )

        return (
            run.get(
                "status"
            )
            == "ok"
            and paths[
                "presence"
            ].exists()
            and paths[
                "detections"
            ].exists()
        )

    except Exception:
        return False

## 14. Run field inference

This is the long cell.

Every successfully completed file is checkpointed immediately.

In [20]:
# ============================================================
# FIX CuPy para CUDA 11.8 en Windows
# ============================================================

import sys
import subprocess

print("Eliminando instalaciones CuPy incompatibles...")

packages_to_remove = [
    "cupy",
    "cupy-cuda12x",
    "cupy-cuda13x",
    "cupy-cuda11x",
]

for package in packages_to_remove:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "uninstall",
            "-y",
            package,
        ],
        check=False,
    )

print()
print("Instalando CuPy compatible con CUDA 11.8...")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "cupy-cuda11x==13.6.0",
    ]
)

print()
print("✅ Instalación terminada.")
print("⚠️ AHORA REINICIA EL KERNEL DEL NOTEBOOK.")

Eliminando instalaciones CuPy incompatibles...

Instalando CuPy compatible con CUDA 11.8...

✅ Instalación terminada.
⚠️ AHORA REINICIA EL KERNEL DEL NOTEBOOK.


In [18]:
import cupy as cp
from cupyx.scipy.signal import fftconvolve

print("CuPy:", cp.__version__)
print("CUDA runtime:", cp.cuda.runtime.runtimeGetVersion())
print("GPUs:", cp.cuda.runtime.getDeviceCount())

props = cp.cuda.runtime.getDeviceProperties(0)
name = props["name"]

if isinstance(name, bytes):
    name = name.decode()

print("GPU:", name)

# prueba real, no solo detección
x = cp.arange(1_000_000, dtype=cp.float64)
y = cp.sum(x * x)

print("Prueba GPU:", float(y.get()))
print("✅ CUDA + CuPy funcionando correctamente")

CuPy: 13.6.0
CUDA runtime: 11080
GPUs: 1
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Prueba GPU: 3.333328333335e+17
✅ CUDA + CuPy funcionando correctamente


In [19]:
# ============================================================
# GPU opcional para la segmentación del notebook 09
# PONER ESTA CELDA JUSTO ANTES DE: "14. Run field inference"
#
# Si encuentra CUDA + CuPy:
#   FIR -> GPU
#   STE -> GPU
#   Moving Average -> GPU
#   log-energy -> GPU
#
# Si no encuentra GPU/CuPy:
#   conserva automáticamente las funciones CPU originales.
#
# Mantiene float64 para no cambiar innecesariamente los resultados.
# ============================================================

# Guardar las implementaciones CPU originales una sola vez.
if "_CPU_bandpass_for_segmentation" not in globals():
    _CPU_bandpass_for_segmentation = bandpass_for_segmentation
    _CPU_short_time_energy = short_time_energy
    _CPU_moving_average_same = moving_average_same
    _CPU_energy_to_db = energy_to_db


USE_GPU = False
GPU_NAME = None

try:
    import cupy as cp
    from cupyx.scipy.signal import fftconvolve as gpu_fftconvolve

    if cp.cuda.runtime.getDeviceCount() > 0:
        USE_GPU = True

        device_id = cp.cuda.Device().id

        GPU_NAME = (
            cp.cuda.runtime
            .getDeviceProperties(device_id)["name"]
        )

        if isinstance(GPU_NAME, bytes):
            GPU_NAME = GPU_NAME.decode()

except Exception as exc:
    USE_GPU = False
    print("GPU/CuPy no disponible:", repr(exc))


if USE_GPU:

    print("=" * 60)
    print("GPU ACTIVADA")
    print("Dispositivo:", GPU_NAME)
    print("=" * 60)

    # --------------------------------------------------------
    # Cachés GPU
    # --------------------------------------------------------

    _GPU_FIR_CACHE = {}

    # Mantener el bloque actual en GPU para que las 9 bandas
    # no vuelvan a copiar los mismos ~30 s desde CPU.
    _GPU_LAST_CPU_BLOCK = None
    _GPU_LAST_BLOCK = None


    def _get_gpu_audio_block(audio):
        global _GPU_LAST_CPU_BLOCK
        global _GPU_LAST_BLOCK

        # Mismo objeto NumPy durante las 9 especies del bloque.
        if audio is not _GPU_LAST_CPU_BLOCK:

            _GPU_LAST_CPU_BLOCK = audio

            _GPU_LAST_BLOCK = cp.asarray(
                np.asarray(
                    audio,
                    dtype=np.float64,
                )
            )

        return _GPU_LAST_BLOCK


    # --------------------------------------------------------
    # Reemplazo GPU: FIR
    # --------------------------------------------------------

    def bandpass_for_segmentation(
        audio,
        sr,
        low_hz,
        high_hz,
    ):
        key = (
            int(sr),
            float(low_hz),
            float(high_hz),
            int(FIR_NUMTAPS),
        )

        # FIR se diseña una sola vez.
        if key not in _GPU_FIR_CACHE:

            # Reutilizar el FIR CPU si ya existe.
            if key not in _FIR_CACHE:
                _FIR_CACHE[key] = firwin(
                    numtaps=FIR_NUMTAPS,
                    cutoff=[
                        low_hz,
                        high_hz,
                    ],
                    pass_zero=False,
                    fs=sr,
                )

            _GPU_FIR_CACHE[key] = cp.asarray(
                _FIR_CACHE[key],
                dtype=cp.float64,
            )

        gpu_audio = _get_gpu_audio_block(
            audio
        )

        return gpu_fftconvolve(
            gpu_audio,
            _GPU_FIR_CACHE[key],
            mode="same",
        )


    # --------------------------------------------------------
    # Reemplazo GPU: STE
    # --------------------------------------------------------

    def short_time_energy(
        audio,
        sr,
    ):
        # Si por alguna razón llega NumPy, usar CPU.
        if not isinstance(
            audio,
            cp.ndarray,
        ):
            return _CPU_short_time_energy(
                audio,
                sr,
            )

        frame_samples = int(
            round(
                sr
                * STE_FRAME_SECONDS
            )
        )

        n_frames = (
            int(audio.size)
            // frame_samples
        )

        if n_frames == 0:
            return (
                cp.empty(
                    0,
                    dtype=cp.float64,
                ),
                frame_samples,
            )

        trimmed = audio[
            :
            n_frames
            * frame_samples
        ]

        frames = trimmed.reshape(
            n_frames,
            frame_samples,
        )

        energy = cp.sum(
            frames
            * frames,
            axis=1,
        )

        return (
            energy,
            frame_samples,
        )


    # --------------------------------------------------------
    # Reemplazo GPU: moving average MA=12
    # --------------------------------------------------------

    def moving_average_same(
        values,
    ):
        if not isinstance(
            values,
            cp.ndarray,
        ):
            return _CPU_moving_average_same(
                values
            )

        if values.size == 0:
            return values.copy()

        kernel = (
            cp.ones(
                STE_MA_LENGTH,
                dtype=cp.float64,
            )
            / STE_MA_LENGTH
        )

        return cp.convolve(
            values,
            kernel,
            mode="same",
        )


    # --------------------------------------------------------
    # Reemplazo GPU: conversión a dB
    #
    # AQUÍ regresamos a NumPy porque ste_db tiene solo ~3000
    # valores por bloque. El pesado FIR/STE ya ocurrió en GPU.
    # Así endpoint_threshold_db y find_endpoint_runs quedan
    # exactamente como estaban.
    # --------------------------------------------------------

    def energy_to_db(
        energy,
    ):
        if not isinstance(
            energy,
            cp.ndarray,
        ):
            return _CPU_energy_to_db(
                energy
            )

        tiny = np.finfo(
            np.float64
        ).tiny

        energy_db_gpu = (
            10.0
            * cp.log10(
                cp.maximum(
                    energy,
                    tiny,
                )
            )
        )

        return cp.asnumpy(
            energy_db_gpu
        )


    # Limpiar memoria GPU retenida de ejecuciones anteriores.
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()

    print(
        "Funciones de segmentación sustituidas "
        "por versiones GPU."
    )


else:

    # Restaurar CPU si se vuelve a ejecutar la celda
    # en una máquina sin CUDA.
    bandpass_for_segmentation = (
        _CPU_bandpass_for_segmentation
    )

    short_time_energy = (
        _CPU_short_time_energy
    )

    moving_average_same = (
        _CPU_moving_average_same
    )

    energy_to_db = (
        _CPU_energy_to_db
    )

    print("=" * 60)
    print("GPU NO DISPONIBLE")
    print("Se continuará con CPU.")
    print("=" * 60)

GPU ACTIVADA
Dispositivo: NVIDIA GeForce RTX 4060 Laptop GPU
Funciones de segmentación sustituidas por versiones GPU.


In [20]:
file_run_rows = []

if RUN_FIELD_SCAN:

    files_to_process = (
        eligible_inventory
        if MAX_FIELD_FILES is None
        else eligible_inventory.head(
            int(
                MAX_FIELD_FILES
            )
        )
    )

    for file_number, row in enumerate(
        files_to_process.itertuples(
            index=False
        ),
        start=1,
    ):
        sample_id = str(
            row.sample_id
        )

        path = Path(
            row.path
        )

        print()
        print(
            f"[{file_number:03d}/{len(files_to_process):03d}] "
            f"{sample_id}"
        )

        paths = checkpoint_paths(
            sample_id
        )

        if (
            RESUME_COMPLETED_FILES
            and checkpoint_complete(
                sample_id
            )
        ):
            print("  checkpoint already complete, skipping")

            with open(
                paths[
                    "run"
                ],
                "r",
                encoding="utf-8",
            ) as f:
                prior_run = json.load(
                    f
                )

            file_run_rows.append(
                prior_run
            )

            continue

        run_row = {
            "sample_id":
                sample_id,
            "source_path":
                str(
                    path
                ),
            "pond":
                row.pond,
            "month":
                row.month,
            "duration_seconds":
                float(
                    row.duration_seconds
                ),
            "status":
                "error",
            "n_raw_candidates":
                np.nan,
            "n_candidates":
                np.nan,
            "n_scored":
                np.nan,
            "n_accepted_raw":
                np.nan,
            "n_accepted_deduplicated":
                np.nan,
            "error":
                None,
        }

        try:
            raw_candidates, candidates = (
                segment_field_file(
                    path
                )
            )

            detections = (
                score_field_candidates(
                    path,
                    candidates,
                )
                if not candidates.empty
                else pd.DataFrame()
            )

            deduplicated = (
                deduplicate_accepted_detections(
                    detections
                )
            )

            presence = pd.DataFrame([
                presence_from_detections(
                    sample_id,
                    detections,
                )
            ])

            for frame in (
                raw_candidates,
                candidates,
                detections,
                deduplicated,
            ):
                if not frame.empty:
                    frame.insert(
                        0,
                        "sample_id",
                        sample_id,
                    )

            raw_candidates.to_csv(
                paths[
                    "raw_candidates"
                ],
                index=False,
            )

            candidates.to_csv(
                paths[
                    "candidates"
                ],
                index=False,
            )

            detections.to_csv(
                paths[
                    "detections"
                ],
                index=False,
            )

            deduplicated.to_csv(
                paths[
                    "accepted_deduplicated"
                ],
                index=False,
            )

            presence.to_csv(
                paths[
                    "presence"
                ],
                index=False,
            )

            run_row.update({
                "status":
                    "ok",
                "n_raw_candidates":
                    int(
                        len(
                            raw_candidates
                        )
                    ),
                "n_candidates":
                    int(
                        len(
                            candidates
                        )
                    ),
                "n_scored":
                    int(
                        len(
                            detections
                        )
                    ),
                "n_accepted_raw":
                    int(
                        detections[
                            "accepted"
                        ].astype(bool).sum()
                    )
                    if not detections.empty
                    else 0,
                "n_accepted_deduplicated":
                    int(
                        len(
                            deduplicated
                        )
                    ),
                "error":
                    None,
            })

            with open(
                paths[
                    "run"
                ],
                "w",
                encoding="utf-8",
            ) as f:
                json.dump(
                    run_row,
                    f,
                    indent=2,
                )

            print(
                "  raw candidates:",
                run_row[
                    "n_raw_candidates"
                ],
                "| merged candidates:",
                run_row[
                    "n_candidates"
                ],
                "| accepted:",
                run_row[
                    "n_accepted_raw"
                ],
            )

        except Exception as exc:
            run_row[
                "error"
            ] = repr(
                exc
            )

            with open(
                paths[
                    "run"
                ],
                "w",
                encoding="utf-8",
            ) as f:
                json.dump(
                    run_row,
                    f,
                    indent=2,
                    default=str,
                )

            print(
                "  ERROR:",
                repr(
                    exc
                ),
            )

        file_run_rows.append(
            run_row
        )

field_run_log = pd.DataFrame(
    file_run_rows
)

field_run_log.to_csv(
    RESULTS_DIR
    / "field_run_log.csv",
    index=False,
)

if RUN_FIELD_SCAN:
    display(
        field_run_log.head(
            30
        )
    )

    print()
    print(
        "Successful files:",
        int(
            (
                field_run_log[
                    "status"
                ]
                == "ok"
            ).sum()
        ),
        "/",
        len(
            field_run_log
        ),
    )


[001/546] qcaz-cc126__A__cc126A- poza1_SP1
  checkpoint already complete, skipping

[002/546] qcaz-cc126__A__cc126A- poza1_SP1A
  checkpoint already complete, skipping

[003/546] qcaz-cc126__A__cc126A- poza1_SP1B
  checkpoint already complete, skipping

[004/546] qcaz-cc126__A__cc126A- poza1_SP3
  checkpoint already complete, skipping

[005/546] qcaz-cc126__A__cc126A- poza1_SP3A
  checkpoint already complete, skipping

[006/546] qcaz-cc126__A__cc126A- poza1_SP3B
  checkpoint already complete, skipping

[007/546] qcaz-cc126__A__cc126A- poza1_SP3C
  checkpoint already complete, skipping

[008/546] qcaz-cc126__A__cc126A- poza2_SP2
  checkpoint already complete, skipping

[009/546] qcaz-cc126__A__cc126A- poza2_SP2A
  checkpoint already complete, skipping

[010/546] qcaz-cc126__A__cc126A- poza2_SP2B
  checkpoint already complete, skipping

[011/546] qcaz-cc126__A__cc126A
  checkpoint already complete, skipping

[012/546] qcaz-cc126__B__cc126B
  checkpoint already complete, skipping

[013/5

,sample_id,source_path,pond,month,duration_seconds,status,n_raw_candidates,n_candidates,n_scored,n_accepted_raw,n_accepted_deduplicated,error
0,qcaz-cc126__A__cc126A- poza1_SP1,C:\Users\sofia\Projects\PUCE\frog-presence-rep...,P1,unknown,599.741917,ok,2720,2701,2701,775,254,None
1,qcaz-cc126__A__cc126A- poza1_SP1A,C:\Users\sofia\Projects\PUCE\frog-presence-rep...,P1,unknown,204.750583,ok,799,785,785,222,53,None
2,qcaz-cc126__A__cc126A- poza1_SP1B,C:\Users\sofia\Projects\PUCE\frog-presence-rep...,P1,unknown,64.488375,ok,346,344,344,34,17,None
3,qcaz-cc126__A__cc126A- poza1_SP3,C:\Users\sofia\Projects\PUCE\frog-presence-rep...,P1,unknown,546.539000,ok,6807,6738,6738,3849,1245,None
4,qcaz-cc126__A__cc126A- poza1_SP3A,C:\Users\sofia\Projects\PUCE\frog-presence-rep...,P1,unknown,33.451542,ok,494,494,494,323,93,None
5,qcaz-cc126__A__cc126A- poza1_SP3B,C:\Users\sofia\Projects\PUCE\frog-presence-rep...,P1,unknown,45.749896,ok,552,550,550,334,103,None
6,qcaz-cc126__A__cc126A- poza1_SP3C,C:\Users\sofia\Projects\PUCE\frog-presence-rep...,P1,unknown,51.653104,ok,635,631,631,476,141,None
7,qcaz-cc126__A__cc126A- poza2_SP2,C:\Users\sofia\Projects\PUCE\frog-presence-rep...,P2,unknown,595.175208,ok,1700,1693,1693,498,150,None
8,qcaz-cc126__A__cc126A- poza2_SP2A,C:\Users\sofia\Projects\PUCE\frog-presence-rep...,P2,unknown,93.297729,ok,360,353,353,14,11,None
9,qcaz-cc126__A__cc126A- poza2_SP2B,C:\Users\sofia\Projects\PUCE\frog-presence-rep...,P2,unknown,63.270875,ok,214,214,214,138,27,None



Successful files: 546 / 546


## 15. Assemble the field presence table from checkpoints

In [21]:
presence_blocks = []

if RUN_FIELD_SCAN:

    successful_ids = (
        field_run_log.loc[
            field_run_log[
                "status"
            ] == "ok",
            "sample_id",
        ]
        .astype(str)
        .tolist()
    )

    for sample_id in successful_ids:
        path = checkpoint_paths(
            sample_id
        )[
            "presence"
        ]

        presence_blocks.append(
            pd.read_csv(
                path
            )
        )

field_presence = (
    pd.concat(
        presence_blocks,
        ignore_index=True,
    )
    if presence_blocks
    else pd.DataFrame()
)

if not field_presence.empty:
    metadata_cols = (
        eligible_inventory[
            [
                "sample_id",
                "relative_path",
                "pond",
                "month",
                "duration_seconds",
            ]
        ]
    )

    field_presence = (
        metadata_cols.merge(
            field_presence,
            on="sample_id",
            how="inner",
            validate="one_to_one",
        )
    )

    for code_name in CLASSES:
        col = (
            f"presence_{code_name}"
        )

        field_presence[
            col
        ] = (
            field_presence[
                col
            ]
            .astype(
                "Int64"
            )
        )

    field_presence[
        "presence_f09"
    ] = pd.array(
        [
            pd.NA
        ]
        * len(
            field_presence
        ),
        dtype="Int64",
    )

    field_presence.to_csv(
        RESULTS_DIR
        / "field_presence_absence.csv",
        index=False,
    )

    display(
        field_presence.head(
            30
        )
    )

    print(
        "Presence rows:",
        len(
            field_presence
        ),
    )

,sample_id,relative_path,pond,month,duration_seconds,presence_f01,presence_f02,presence_f03,presence_f04,presence_f05,presence_f06,presence_f07,presence_f08,presence_f09,presence_f10
0,qcaz-cc126__A__cc126A- poza1_SP1,qcaz-cc126\A\cc126A- poza1_SP1.wav,P1,unknown,599.741917,0,1,1,1,1,1,1,0,<NA>,1
1,qcaz-cc126__A__cc126A- poza1_SP1A,qcaz-cc126\A\cc126A- poza1_SP1A.wav,P1,unknown,204.750583,0,1,1,1,0,1,1,0,<NA>,0
2,qcaz-cc126__A__cc126A- poza1_SP1B,qcaz-cc126\A\cc126A- poza1_SP1B.wav,P1,unknown,64.488375,0,1,1,0,0,1,1,0,<NA>,0
3,qcaz-cc126__A__cc126A- poza1_SP3,qcaz-cc126\A\cc126A- poza1_SP3.wav,P1,unknown,546.539000,0,1,1,0,1,1,1,0,<NA>,0
4,qcaz-cc126__A__cc126A- poza1_SP3A,qcaz-cc126\A\cc126A- poza1_SP3A.wav,P1,unknown,33.451542,0,1,1,0,0,1,0,0,<NA>,0
5,qcaz-cc126__A__cc126A- poza1_SP3B,qcaz-cc126\A\cc126A- poza1_SP3B.wav,P1,unknown,45.749896,0,1,1,0,1,1,0,0,<NA>,0
6,qcaz-cc126__A__cc126A- poza1_SP3C,qcaz-cc126\A\cc126A- poza1_SP3C.wav,P1,unknown,51.653104,0,1,1,0,0,1,1,0,<NA>,0
7,qcaz-cc126__A__cc126A- poza2_SP2,qcaz-cc126\A\cc126A- poza2_SP2.wav,P2,unknown,595.175208,0,1,1,1,1,1,1,0,<NA>,1
8,qcaz-cc126__A__cc126A- poza2_SP2A,qcaz-cc126\A\cc126A- poza2_SP2A.wav,P2,unknown,93.297729,0,1,0,1,0,1,0,0,<NA>,1
9,qcaz-cc126__A__cc126A- poza2_SP2B,qcaz-cc126\A\cc126A- poza2_SP2B.wav,P2,unknown,63.270875,0,1,1,0,0,1,1,0,<NA>,1


Presence rows: 546


## 16. Build global event tables

Raw per-file checkpoints remain the primary audit trail.

This optional consolidation is convenient for analysis.

In [22]:
field_candidates = pd.DataFrame()
field_detections = pd.DataFrame()
field_accepted_deduplicated = pd.DataFrame()

if (
    RUN_FIELD_SCAN
    and BUILD_GLOBAL_EVENT_TABLES
):

    candidate_blocks = []
    detection_blocks = []
    accepted_blocks = []

    for sample_id in (
        field_run_log.loc[
            field_run_log[
                "status"
            ] == "ok",
            "sample_id",
        ]
        .astype(str)
    ):
        paths = checkpoint_paths(
            sample_id
        )

        for (
            path_key,
            destination,
        ) in (
            (
                "candidates",
                candidate_blocks,
            ),
            (
                "detections",
                detection_blocks,
            ),
            (
                "accepted_deduplicated",
                accepted_blocks,
            ),
        ):
            path = paths[
                path_key
            ]

            if (
                path.exists()
                and path.stat().st_size
                > 0
            ):
                try:
                    frame = pd.read_csv(
                        path
                    )

                    if not frame.empty:
                        destination.append(
                            frame
                        )
                except pd.errors.EmptyDataError:
                    pass

    field_candidates = (
        pd.concat(
            candidate_blocks,
            ignore_index=True,
        )
        if candidate_blocks
        else pd.DataFrame()
    )

    field_detections = (
        pd.concat(
            detection_blocks,
            ignore_index=True,
        )
        if detection_blocks
        else pd.DataFrame()
    )

    field_accepted_deduplicated = (
        pd.concat(
            accepted_blocks,
            ignore_index=True,
        )
        if accepted_blocks
        else pd.DataFrame()
    )

    field_candidates.to_csv(
        RESULTS_DIR
        / "field_candidates.csv",
        index=False,
    )

    field_detections.to_csv(
        RESULTS_DIR
        / "field_detector_decisions.csv",
        index=False,
    )

    field_accepted_deduplicated.to_csv(
        RESULTS_DIR
        / "field_accepted_detections_deduplicated.csv",
        index=False,
    )

    print("Global candidates :", len(field_candidates))
    print("Global decisions  :", len(field_detections))
    print(
        "Deduplicated accepted:",
        len(
            field_accepted_deduplicated
        ),
    )

Global candidates : 1376119
Global decisions  : 1376119
Deduplicated accepted: 231058


## 17. Presence counts by species

In [23]:
if not field_presence.empty:

    presence_summary_rows = []

    for code_name in PAPER_CLASS_ORDER:
        column = (
            f"presence_{code_name}"
        )

        if code_name not in CLASSES:
            presence_summary_rows.append({
                "code":
                    code_name,
                "modelled":
                    False,
                "n_samples":
                    len(
                        field_presence
                    ),
                "n_present":
                    np.nan,
                "presence_fraction":
                    np.nan,
            })

            continue

        values = (
            field_presence[
                column
            ].astype(
                "Int64"
            )
        )

        presence_summary_rows.append({
            "code":
                code_name,
            "modelled":
                True,
            "n_samples":
                len(
                    values
                ),
            "n_present":
                int(
                    values.sum()
                ),
            "presence_fraction":
                float(
                    values.mean()
                ),
        })

    presence_summary = pd.DataFrame(
        presence_summary_rows
    )

    presence_summary.to_csv(
        RESULTS_DIR
        / "presence_summary_by_species.csv",
        index=False,
    )

    display(
        presence_summary.round(4)
    )

,code,modelled,n_samples,n_present,presence_fraction
0,f01,True,546,37.0,0.0678
1,f02,True,546,456.0,0.8352
2,f03,True,546,229.0,0.4194
3,f04,True,546,163.0,0.2985
4,f05,True,546,130.0,0.2381
5,f06,True,546,259.0,0.4744
6,f07,True,546,119.0,0.2179
7,f08,True,546,80.0,0.1465
8,f09,False,546,NaN,NaN
9,f10,True,546,62.0,0.1136


## 18. Presence by pond and month

These are descriptive summaries of **our detector outputs**, not ground truth.

In [24]:
if not field_presence.empty:

    modelled_presence_columns = [
        f"presence_{code_name}"
        for code_name in CLASSES
    ]

    pond_summary = (
        field_presence.groupby(
            "pond",
            dropna=False,
        )[
            modelled_presence_columns
        ]
        .mean()
        .reset_index()
    )

    month_summary = (
        field_presence.groupby(
            "month",
            dropna=False,
        )[
            modelled_presence_columns
        ]
        .mean()
        .reset_index()
    )

    pond_summary.to_csv(
        RESULTS_DIR
        / "presence_fraction_by_pond.csv",
        index=False,
    )

    month_summary.to_csv(
        RESULTS_DIR
        / "presence_fraction_by_month.csv",
        index=False,
    )

    print("By pond:")
    display(
        pond_summary.round(3)
    )

    print("By month:")
    display(
        month_summary.round(3)
    )

By pond:


,pond,presence_f01,presence_f02,presence_f03,presence_f04,presence_f05,presence_f06,presence_f07,presence_f08,presence_f10
0,P1,0.0,1.0,1.0,0.286,0.429,1.0,0.714,0.0,0.143
1,P2,0.0,1.0,0.667,0.667,0.333,1.0,0.667,0.0,1.0
2,unknown,0.069,0.832,0.41,0.297,0.235,0.465,0.209,0.149,0.108


By month:


,month,presence_f01,presence_f02,presence_f03,presence_f04,presence_f05,presence_f06,presence_f07,presence_f08,presence_f10
0,unknown,0.068,0.835,0.419,0.299,0.238,0.474,0.218,0.147,0.114


## 19. Detector-output sanity checks

These checks look for pathological field behavior. They do not constitute test accuracy.

In [25]:
if not field_presence.empty:

    for code_name in CLASSES:
        column = (
            f"presence_{code_name}"
        )

        unique = set(
            field_presence[
                column
            ]
            .dropna()
            .astype(int)
            .unique()
        )

        if not unique.issubset(
            {
                0,
                1,
            }
        ):
            raise AssertionError(
                f"Non-binary presence values for {code_name}: {unique}"
            )

    if not field_presence[
        "sample_id"
    ].is_unique:
        raise AssertionError(
            "Presence sample_id must be unique."
        )

    if not field_presence[
        "presence_f09"
    ].isna().all():
        raise AssertionError(
            "Unmodelled f09 must remain NA."
        )

    print("Binary presence for modelled classes : OK")
    print("Unique sample IDs                    : OK")
    print("f09 is NA, not zero                 : OK")

if (
    not field_detections.empty
):
    valid = field_detections[
        field_detections[
            "feature_valid"
        ].astype(bool)
    ]

    print()
    print(
        "Valid feature fraction:",
        round(
            field_detections[
                "feature_valid"
            ].astype(bool).mean(),
            4,
        ),
    )

    if not valid.empty:
        print(
            "Raw accepted fraction:",
            round(
                valid[
                    "accepted"
                ].astype(bool).mean(),
                4,
            ),
        )

        print(
            "Median candidate duration:",
            round(
                valid[
                    "duration_seconds"
                ].median(),
                4,
            ),
            "s",
        )

Binary presence for modelled classes : OK
Unique sample IDs                    : OK
f09 is NA, not zero                 : OK

Valid feature fraction: 1.0
Raw accepted fraction: 0.593
Median candidate duration: 0.17 s


## 20. Accepted detections by predicted species and source segmentation band

This table is useful for spotting an alarming failure mode: one broad frequency band generating nearly
all detections for an unrelated class.

In [26]:
if (
    not field_detections.empty
    and "accepted"
    in field_detections.columns
):
    accepted_raw = (
        field_detections[
            field_detections[
                "accepted"
            ].astype(bool)
        ]
        .copy()
    )

    if not accepted_raw.empty:
        accepted_cross_tab = pd.crosstab(
            accepted_raw[
                "segmentation_code"
            ],
            accepted_raw[
                "prediction"
            ],
        )

        accepted_cross_tab.to_csv(
            RESULTS_DIR
            / "accepted_detection_band_vs_prediction.csv"
        )

        display(
            accepted_cross_tab
        )

prediction,f01,f02,f03,f04,f05,f06,f07,f08,f10
segmentation_code,,,,,,,,,
f01,222,78018,22784,995,733,3900,334,177,69
f02,665,73195,19464,408,89,3608,27,106,30
f03,124,39278,19165,1256,234,1252,281,69,192
f04,21,62025,10099,845,2407,3118,190,36,34
f05,629,77036,21893,540,225,3749,305,167,44
f06,113,66731,21926,1031,1075,1830,290,59,108
f07,159,77052,23791,1020,931,2306,530,80,321
f08,323,24922,30184,291,34,234,311,180,113
f10,169,79372,24819,832,452,3419,590,160,281


## 21. Detection LLR summaries in field audio

Large differences from calibration are expected because pond recordings contain environmental noise,
overlaps and unmodelled species. The table is diagnostic only.

In [27]:
if (
    not field_detections.empty
    and "hypothesis"
    in field_detections.columns
):

    llr_field_summary = (
        field_detections[
            field_detections[
                "feature_valid"
            ].astype(bool)
        ]
        .groupby(
            "hypothesis"
        )
        .agg(
            n=(
                "llr",
                "size",
            ),
            median_llr=(
                "llr",
                "median",
            ),
            q10_llr=(
                "llr",
                lambda x:
                    x.quantile(
                        0.10
                    ),
            ),
            q90_llr=(
                "llr",
                lambda x:
                    x.quantile(
                        0.90
                    ),
            ),
            acceptance_fraction=(
                "accepted",
                "mean",
            ),
        )
        .reset_index()
    )

    llr_field_summary.to_csv(
        RESULTS_DIR
        / "field_llr_summary_by_hypothesis.csv",
        index=False,
    )

    display(
        llr_field_summary.round(4)
    )

,hypothesis,n,median_llr,q10_llr,q90_llr,acceptance_fraction
0,f01,3550,9.8860,3.8942,14.4285,0.6831
1,f02,638803,6.7390,3.7459,11.7176,0.9042
2,f03,455382,6.3673,3.5352,12.7365,0.4263
3,f04,68115,3.6511,2.5435,5.1153,0.1060
4,f05,11718,5.7600,2.9033,10.1274,0.5274
5,f06,138316,3.7970,2.4188,5.9747,0.1693
6,f07,14359,3.5132,1.9258,7.4035,0.1990
7,f08,3587,5.2489,2.3778,9.9614,0.2883
8,f10,42289,3.8883,2.3029,6.5196,0.0282


## 22. Paper-reported field metrics

The article manually labelled a subset of **18** field samples and reports final presence/absence
performance approximately as:

- Recall: **0.875**
- Precision: **1.000**
- F1: **0.933**
- MCC: **0.914**
- Specificity: **1.000**
- Accuracy: **0.966**

Those are **paper results**, not metrics produced by this notebook.

Our field scan must not be compared numerically against those values unless complete, explicit
ground-truth vectors for the corresponding samples are available. Partial species notes do not justify
turning every unmentioned species into a negative label.

In [28]:
paper_field_metrics = pd.DataFrame([
    {
        "metric":
            "Recall",
        "paper_value":
            0.875,
    },
    {
        "metric":
            "Precision",
        "paper_value":
            1.000,
    },
    {
        "metric":
            "F1",
        "paper_value":
            0.933,
    },
    {
        "metric":
            "MCC",
        "paper_value":
            0.914,
    },
    {
        "metric":
            "Specificity",
        "paper_value":
            1.000,
    },
    {
        "metric":
            "Accuracy",
        "paper_value":
            0.966,
    },
])

display(
    paper_field_metrics
)

,metric,paper_value
0,Recall,0.875
1,Precision,1.000
2,F1,0.933
3,MCC,0.914
4,Specificity,1.000
5,Accuracy,0.966


## 23. Run manifest

In [29]:
run_manifest = {
    "source_notebook":
        "09_field_presence_absence",
    "conceptual_stage":
        "final field presence-absence stage of source paper",
    "dataset":
        "Dataset 2 pond recordings",
    "pond_root":
        str(
            PONDS_CALL_DIR
        ),
    "eligible_wav_count":
        int(
            len(
                eligible_inventory
            )
        ),
    "processed_wav_count":
        int(
            (
                field_run_log[
                    "status"
                ]
                == "ok"
            ).sum()
        )
        if RUN_FIELD_SCAN
        and not field_run_log.empty
        else 0,
    "detector_manifest":
        str(
            FINAL_DETECTOR_MANIFEST_PATH
        ),
    "segmentation_configuration":
        str(
            FINAL_SEGMENTATION_CONFIG_PATH
        ),
    "detector_feature":
        detector_manifest[
            "feature"
        ],
    "detector_n_components":
        detector_manifest[
            "n_components"
        ],
    "classes_modelled":
        list(
            CLASSES
        ),
    "f09_status":
        "NA_unmodelled",
    "presence_rule":
        "one or more accepted detections in sample => presence",
    "thresholds_retuned_on_ponds":
        False,
    "segmentation_retuned_on_ponds":
        False,
    "field_metrics_computed":
        False,
    "field_metric_reason":
        (
            "complete explicit ground-truth vectors for the paper's "
            "18 manually labelled samples are not assumed available"
        ),
}

with open(
    RESULTS_DIR
    / "field_inference_manifest.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        run_manifest,
        f,
        indent=2,
    )

print(
    json.dumps(
        run_manifest,
        indent=2,
    )
)

{
  "source_notebook": "09_field_presence_absence",
  "conceptual_stage": "final field presence-absence stage of source paper",
  "dataset": "Dataset 2 pond recordings",
  "pond_root": "C:\\Users\\sofia\\Projects\\PUCE\\frog-presence-replication\\data\\pond-recordings",
  "eligible_wav_count": 546,
  "processed_wav_count": 546,
  "detector_manifest": "C:\\Users\\sofia\\Projects\\PUCE\\frog-presence-replication\\models\\gmm_robust_final\\final_detector_manifest.json",
  "segmentation_configuration": "C:\\Users\\sofia\\Projects\\PUCE\\frog-presence-replication\\data\\processed\\candidate_call_segmentation\\final_segmentation_configuration.json",
  "detector_feature": "20_PLP_RASTA",
  "detector_n_components": 4,
  "classes_modelled": [
    "f01",
    "f02",
    "f03",
    "f04",
    "f05",
    "f06",
    "f07",
    "f08",
    "f10"
  ],
  "f09_status": "NA_unmodelled",
  "presence_rule": "one or more accepted detections in sample => presence",
  "thresholds_retuned_on_ponds": false,
  "s

## 24. Final interpretation

After a successful full run, inspect in this order:

1. `field_run_log`
2. `field_presence`
3. `presence_summary`
4. `accepted_cross_tab`
5. `llr_field_summary`

### What counts as success

A successful execution means:

- the recovered Dataset-2 inventory has been explicitly audited;
- all intended eligible files are processed without hidden failures;
- frozen segmentation and frozen detector artifacts are used unchanged;
- accepted calls are aggregated to presence/absence;
- `f09` remains `NA`;
- results are saved with per-file audit trails.

### What does **not** count as success

A large number of detections is not automatically evidence that the system works well. Pond audio
contains unmodelled frogs, insects, human sounds, mechanical artifacts and environmental noise. The
verification layer is supposed to reject much of that, but independent labels are required to quantify
how well it actually does so.

### Conceptual status of the paper replication

With this notebook, the full conceptual pipeline of the source article has been implemented:

**labelled calls → acoustic features → GMM species models → likelihood-ratio verification → automatic
call segmentation → field presence/absence.**

Anything after this point is evaluation, ecological summarization, error analysis, or methodological
extension rather than another missing major block of the original pipeline.